# House Prices competition
> Author | Lavrov Evgeniy - ResInfesser

Models: catboost, xgboost, lightgbm

## 1. Import Libraries & Load Data

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error

/home/koting/_code/Kaggle-Solutions/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.competition_download("house-prices-advanced-regression-techniques")
train_df = pd.read_csv(f"{path}/train.csv")
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
test_df = pd.read_csv(f"{path}/test.csv")
test_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


## 2. Processing dataframes

In [4]:
train_df = train_df.copy()

train_df = train_df[(train_df['SalePrice'] > 50000) & (train_df['SalePrice'] < 600000)]

if 'Id' in train_df.columns:
    train_df = train_df.drop('Id', axis=1)

y_train = np.log1p(train_df['SalePrice'])
train_df = train_df.drop('SalePrice', axis=1)

for col in ['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
            'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
            'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType']:
    if col in train_df.columns:
        train_df[col] = train_df[col].fillna('NA')

mapping = {
    'MSSubClass': {20:2,30:1,40:3,45:4,50:5,60:6,70:4,75:6,80:5,85:5,90:3,120:7,150:7,160:8,180:7,190:2},
    'MSZoning': {'RL':6,'RP':5,'RM':4,'RH':3,'FV':2,'C':1,'I':1,'A':0},
    'Street': {'Pave':1,'Grvl':0},
    'Alley': {'Pave':2,'Grvl':1,'NA':0},
    'LotShape': {'Reg':3,'IR1':2,'IR2':1,'IR3':0},
    'LandContour': {'Lvl':3,'HLS':2,'Bnk':1,'Low':0},
    'Utilities': {'AllPub':3,'NoSewr':2,'NoSeWa':1,'ELO':0},
    'LotConfig': {'FR3':5,'FR2':4,'CulDSac':3,'Corner':2,'Inside':1},
    'LandSlope': {'Gtl':2,'Mod':1,'Sev':0},
    'Neighborhood': {'StoneBr':20,'Somerst':19,'NoRidge':18,'NridgHt':17,'Timber':16,'Veenker':15,'Blmngtn':14,
                     'CollgCr':13,'ClearCr':12,'Edwards':11,'Gilbert':10,'NWAmes':9,'Sawyer':8,'SawyerW':7,
                     'BrkSide':6,'Crawfor':5,'Mitchel':4,'Names':3,'NPkVill':2,'SWISU':2,'MeadowV':1,'OldTown':1,
                     'BrDale':0,'IDOTRR':0},
    'Condition1': {'PosA':5,'PosN':4,'Norm':3,'Feedr':2,'Artery':1,'RRNn':0,'RRAn':0,'RRNe':0,'RRAe':0},
    'Condition2': {'PosA':5,'PosN':4,'Norm':3,'Feedr':2,'Artery':1,'RRNn':0,'RRAn':0,'RRNe':0,'RRAe':0},
    'BldgType': {'1Fam':4,'TwnhsE':3,'TwnhsI':2,'Duplx':1,'2FmCon':0},
    'HouseStyle': {'2.5Fin':7,'2Story':6,'2.5Unf':5,'1.5Fin':4,'1Story':3,'1.5Unf':2,'SFoyer':1,'SLvl':0},
    'RoofStyle': {'Mansard':5,'Hip':4,'Gable':3,'Gambrel':2,'Shed':1,'Flat':0},
    'RoofMatl': {'WdShake':6,'WdShngl':5,'ClyTile':4,'Metal':3,'CompShg':2,'Tar&Grv':1,'Membran':1,'Roll':0},
    'Exterior1st': {'Stone':9,'BrkFace':8,'BrkComm':7,'Stucco':6,'CemntBd':5,'HdBoard':5,'VinylSd':4,'Wd Sdng':4,
                    'WdShing':4,'MetalSd':3,'AsphShn':2,'AsbShng':2,'Plywood':2,'CBlock':1,'ImStucco':1,'Other':0},
    'Exterior2nd': {'Stone':9,'BrkFace':8,'BrkComm':7,'Stucco':6,'CemntBd':5,'HdBoard':5,'VinylSd':4,'Wd Sdng':4,
                    'WdShing':4,'MetalSd':3,'AsphShn':2,'AsbShng':2,'Plywood':2,'CBlock':1,'ImStucco':1,'Other':0},
    'MasVnrType': {'Stone':3,'BrkFace':2,'BrkCmn':1,'CBlock':1,'None':0,'NA':0},
    'Foundation': {'PConc':5,'CBlock':4,'BrkTil':3,'Stone':2,'Slab':1,'Wood':0},
    'Heating': {'GasA':4,'GasW':3,'Grav':2,'Wall':2,'Floor':1,'OthW':0},
    'CentralAir': {'Y':1,'N':0},
    'Electrical': {'SBrkr':3,'FuseA':2,'Mix':1,'FuseF':1,'FuseP':0},
    'Functional': {'Typ':7,'Min1':6,'Min2':5,'Mod':4,'Maj1':3,'Maj2':2,'Sev':1,'Sal':0},
    'GarageType': {'BuiltIn':6,'Attchd':5,'Detchd':4,'Basment':3,'2Types':2,'CarPort':1,'NA':0},
    'GarageFinish': {'Fin':3,'RFn':2,'Unf':1,'NA':0},
    'PavedDrive': {'Y':2,'P':1,'N':0},
    'Fence': {'GdPrv':4,'MnPrv':3,'GdWo':2,'MnWw':1,'NA':0},
    'MiscFeature': {'Elev':5,'TenC':4,'Gar2':3,'Shed':2,'Othr':1,'NA':0},
    'SaleType': {'New':5,'WD':4,'CWD':3,'VWD':2,'COD':1,'Con':1,'ConLw':1,'ConLI':1,'ConLD':1,'Oth':0},
    'SaleCondition': {'Normal':4,'AdjLand':3,'Partial':3,'Abnorml':2,'Family':1,'Alloca':1}
}

qual_cols = ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual','GarageQual','GarageCond','PoolQC','FireplaceQu']
for col in qual_cols:
    mapping[col] = {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'NA':0}

bsmt_cols = ['BsmtExposure','BsmtFinType1','BsmtFinType2']
for col in bsmt_cols:
    if col == 'BsmtExposure':
        mapping[col] = {'Gd':4,'Av':3,'Mn':2,'No':1,'NA':0}
    else:
        mapping[col] = {'GLQ':6,'ALQ':5,'BLQ':4,'Rec':3,'LwQ':2,'Unf':1,'NA':0}

for col, m in mapping.items():
    if col in train_df.columns:
        train_df[col] = train_df[col].map(m)

for col in train_df.select_dtypes(include=['object']).columns:
    train_df[col] = pd.to_numeric(train_df[col], errors='coerce').fillna(0)

train_df['LotFrontage'] = train_df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

train_df['TotalSF'] = train_df['TotalBsmtSF'] + train_df['1stFlrSF'] + train_df['2ndFlrSF']
train_df['TotalBath'] = (train_df['FullBath'] + 0.5*train_df['HalfBath'] + 
                         train_df['BsmtFullBath'] + 0.5*train_df['BsmtHalfBath'])
train_df['Age'] = train_df['YrSold'] - train_df['YearBuilt']
train_df['RemodelAge'] = train_df['YrSold'] - train_df['YearRemodAdd']

X_train = train_df.to_numpy()
X_train_part, X_val, y_train_part, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

In [5]:
test_df = test_df.copy()
ids = test_df['Id'].copy() 
test_df = test_df.drop('Id', axis=1) 

for col in ['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
            'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
            'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType']:
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna('NA')

mapping = {
    'MSSubClass': {20:2,30:1,40:3,45:4,50:5,60:6,70:4,75:6,80:5,85:5,90:3,120:7,150:7,160:8,180:7,190:2},
    'MSZoning': {'RL':6,'RP':5,'RM':4,'RH':3,'FV':2,'C':1,'I':1,'A':0},
    'Street': {'Pave':1,'Grvl':0},
    'Alley': {'Pave':2,'Grvl':1,'NA':0},
    'LotShape': {'Reg':3,'IR1':2,'IR2':1,'IR3':0},
    'LandContour': {'Lvl':3,'HLS':2,'Bnk':1,'Low':0},
    'Utilities': {'AllPub':3,'NoSewr':2,'NoSeWa':1,'ELO':0},
    'LotConfig': {'FR3':5,'FR2':4,'CulDSac':3,'Corner':2,'Inside':1},
    'LandSlope': {'Gtl':2,'Mod':1,'Sev':0},
    'Neighborhood': {'StoneBr':20,'Somerst':19,'NoRidge':18,'NridgHt':17,'Timber':16,'Veenker':15,'Blmngtn':14,
                     'CollgCr':13,'ClearCr':12,'Edwards':11,'Gilbert':10,'NWAmes':9,'Sawyer':8,'SawyerW':7,
                     'BrkSide':6,'Crawfor':5,'Mitchel':4,'Names':3,'NPkVill':2,'SWISU':2,'MeadowV':1,'OldTown':1,
                     'BrDale':0,'IDOTRR':0},
    'Condition1': {'PosA':5,'PosN':4,'Norm':3,'Feedr':2,'Artery':1,'RRNn':0,'RRAn':0,'RRNe':0,'RRAe':0},
    'Condition2': {'PosA':5,'PosN':4,'Norm':3,'Feedr':2,'Artery':1,'RRNn':0,'RRAn':0,'RRNe':0,'RRAe':0},
    'BldgType': {'1Fam':4,'TwnhsE':3,'TwnhsI':2,'Duplx':1,'2FmCon':0},
    'HouseStyle': {'2.5Fin':7,'2Story':6,'2.5Unf':5,'1.5Fin':4,'1Story':3,'1.5Unf':2,'SFoyer':1,'SLvl':0},
    'RoofStyle': {'Mansard':5,'Hip':4,'Gable':3,'Gambrel':2,'Shed':1,'Flat':0},
    'RoofMatl': {'WdShake':6,'WdShngl':5,'ClyTile':4,'Metal':3,'CompShg':2,'Tar&Grv':1,'Membran':1,'Roll':0},
    'Exterior1st': {'Stone':9,'BrkFace':8,'BrkComm':7,'Stucco':6,'CemntBd':5,'HdBoard':5,'VinylSd':4,'Wd Sdng':4,
                    'WdShing':4,'MetalSd':3,'AsphShn':2,'AsbShng':2,'Plywood':2,'CBlock':1,'ImStucco':1,'Other':0},
    'Exterior2nd': {'Stone':9,'BrkFace':8,'BrkComm':7,'Stucco':6,'CemntBd':5,'HdBoard':5,'VinylSd':4,'Wd Sdng':4,
                    'WdShing':4,'MetalSd':3,'AsphShn':2,'AsbShng':2,'Plywood':2,'CBlock':1,'ImStucco':1,'Other':0},
    'MasVnrType': {'Stone':3,'BrkFace':2,'BrkCmn':1,'CBlock':1,'None':0,'NA':0},
    'Foundation': {'PConc':5,'CBlock':4,'BrkTil':3,'Stone':2,'Slab':1,'Wood':0},
    'Heating': {'GasA':4,'GasW':3,'Grav':2,'Wall':2,'Floor':1,'OthW':0},
    'CentralAir': {'Y':1,'N':0},
    'Electrical': {'SBrkr':3,'FuseA':2,'Mix':1,'FuseF':1,'FuseP':0},
    'Functional': {'Typ':7,'Min1':6,'Min2':5,'Mod':4,'Maj1':3,'Maj2':2,'Sev':1,'Sal':0},
    'GarageType': {'BuiltIn':6,'Attchd':5,'Detchd':4,'Basment':3,'2Types':2,'CarPort':1,'NA':0},
    'GarageFinish': {'Fin':3,'RFn':2,'Unf':1,'NA':0},
    'PavedDrive': {'Y':2,'P':1,'N':0},
    'Fence': {'GdPrv':4,'MnPrv':3,'GdWo':2,'MnWw':1,'NA':0},
    'MiscFeature': {'Elev':5,'TenC':4,'Gar2':3,'Shed':2,'Othr':1,'NA':0},
    'SaleType': {'New':5,'WD':4,'CWD':3,'VWD':2,'COD':1,'Con':1,'ConLw':1,'ConLI':1,'ConLD':1,'Oth':0},
    'SaleCondition': {'Normal':4,'AdjLand':3,'Partial':3,'Abnorml':2,'Family':1,'Alloca':1}
}

qual_cols = ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual','GarageQual','GarageCond','PoolQC','FireplaceQu']
for col in qual_cols:
    mapping[col] = {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'NA':0}

bsmt_cols = ['BsmtExposure','BsmtFinType1','BsmtFinType2']
for col in bsmt_cols:
    if col == 'BsmtExposure':
        mapping[col] = {'Gd':4,'Av':3,'Mn':2,'No':1,'NA':0}
    else:
        mapping[col] = {'GLQ':6,'ALQ':5,'BLQ':4,'Rec':3,'LwQ':2,'Unf':1,'NA':0}

for col, m in mapping.items():
    if col in test_df.columns:
        test_df[col] = test_df[col].map(m)

for col in test_df.select_dtypes(include=['object']).columns:
    test_df[col] = pd.to_numeric(test_df[col], errors='coerce').fillna(0)

test_df['LotFrontage'] = test_df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

test_df['TotalSF'] = test_df['TotalBsmtSF'] + test_df['1stFlrSF'] + test_df['2ndFlrSF']
test_df['TotalBath'] = (test_df['FullBath'] + 0.5*test_df['HalfBath'] + 
                         test_df['BsmtFullBath'] + 0.5*test_df['BsmtHalfBath'])
test_df['Age'] = test_df['YrSold'] - test_df['YearBuilt']
test_df['RemodelAge'] = test_df['YrSold'] - test_df['YearRemodAdd']

X_test = test_df.to_numpy()

## 4. Model

### 4.1 Training a model

In [6]:
# Initialize CatBoostRegressor
model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    early_stopping_rounds=50,
    verbose=100,
    eval_metric='RMSE'
)

# Fit model
model.fit(
    X_train_part, y_train_part,
    eval_set=(X_val, y_val),
    verbose=100
)

# Get predictions
final_preds_val = model.predict(X_val)
final_preds_train = model.predict(X_train_part)

0:	learn: 0.3814044	test: 0.3523556	best: 0.3523556 (0)	total: 59.3ms	remaining: 1m 58s
100:	learn: 0.1190166	test: 0.1285473	best: 0.1285473 (100)	total: 660ms	remaining: 12.4s
200:	learn: 0.0870849	test: 0.1172820	best: 0.1172820 (200)	total: 1.25s	remaining: 11.2s
300:	learn: 0.0728607	test: 0.1147851	best: 0.1147748 (298)	total: 1.82s	remaining: 10.3s
400:	learn: 0.0619886	test: 0.1139417	best: 0.1139182 (399)	total: 2.49s	remaining: 9.94s
500:	learn: 0.0523462	test: 0.1133699	best: 0.1132783 (494)	total: 3.08s	remaining: 9.22s
600:	learn: 0.0452806	test: 0.1130325	best: 0.1130161 (598)	total: 3.65s	remaining: 8.5s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.1128918199
bestIteration = 615

Shrink model to first 616 iterations.


### 4.2 Use of metrics to assess quality

In [7]:
print(f"RMSE: {root_mean_squared_error(y_val, final_preds_val):.3f}")
print(f"mean_squared_error: {mean_squared_error(y_val, final_preds_val):.3f}")
print(f"mean_absolute_error: {mean_absolute_error(y_val, final_preds_val):.3f}")
print(f"Validation mean_absolute_percentage_error: {mean_absolute_percentage_error(y_val, final_preds_val):.3f}")
print(f"r2_score: {r2_score(y_val, final_preds_val):.3f}")

RMSE: 0.113
mean_squared_error: 0.013
mean_absolute_error: 0.078
Validation mean_absolute_percentage_error: 0.006
r2_score: 0.901


## 4.3 Retrain for all data

In [8]:
assert X_train.shape[1] == X_test.shape[1]

cat = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    early_stopping_rounds=50,
    verbose=100,
    eval_metric='RMSE'
)
xgb = XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6)
lgb = LGBMRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6, verbose=-1)

cat.fit(X_train, y_train)
xgb.fit(X_train, y_train)
lgb.fit(X_train, y_train)

pred_cat = np.expm1(cat.predict(X_test))
pred_xgb = np.expm1(xgb.predict(X_test))
pred_lgb = np.expm1(lgb.predict(X_test))

final_preds = (pred_cat + pred_xgb + pred_lgb) / 3

0:	learn: 0.3757249	total: 10.2ms	remaining: 20.5s
100:	learn: 0.1182403	total: 659ms	remaining: 12.4s
200:	learn: 0.0889570	total: 1.29s	remaining: 11.6s
300:	learn: 0.0768068	total: 1.9s	remaining: 10.7s
400:	learn: 0.0672179	total: 2.49s	remaining: 9.91s
500:	learn: 0.0580369	total: 3.11s	remaining: 9.32s
600:	learn: 0.0510021	total: 3.73s	remaining: 8.69s
700:	learn: 0.0450721	total: 4.35s	remaining: 8.06s
800:	learn: 0.0401525	total: 4.99s	remaining: 7.47s
900:	learn: 0.0363384	total: 5.68s	remaining: 6.92s
1000:	learn: 0.0329682	total: 6.43s	remaining: 6.41s
1100:	learn: 0.0297108	total: 7.09s	remaining: 5.79s
1200:	learn: 0.0267188	total: 7.73s	remaining: 5.14s
1300:	learn: 0.0238311	total: 8.38s	remaining: 4.5s
1400:	learn: 0.0216488	total: 8.99s	remaining: 3.84s
1500:	learn: 0.0196125	total: 9.6s	remaining: 3.19s
1600:	learn: 0.0177742	total: 10.2s	remaining: 2.55s
1700:	learn: 0.0161650	total: 10.9s	remaining: 1.91s
1800:	learn: 0.0147024	total: 11.6s	remaining: 1.28s
1900:	l

/home/koting/_code/Kaggle-Solutions/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## 5. Final csv

In [ ]:
submission = pd.DataFrame({
    'Id': ids,
    'SalePrice': final_preds
})

submission.to_csv('../submissions/house-prices-submission.csv', index=False)
print(submission.head())

     Id      SalePrice
0  1461  121267.550113
1  1462  160272.744241
2  1463  187476.006513
3  1464  194549.812029
4  1465  186677.692223
